In [1]:
!pip install networkx

import numpy as np
import pandas as pd 
import statsmodels.api as sm
import statsmodels.formula.api as smf
import networkx as nx
from typing import Dict

In [2]:
# DAG final  

dag_final= {
  "noeuds": [
    "Account_Age",
    "Location",
    "Transaction_Type",
    "Device_Used",
    "Number_of_Transactions_Last_24H",
    "Payment_Method",
    "Time_of_Transaction",
    "Fraudulent"
  ],
  "liens": [
    {
      "de": "Location",
      "vers": "Fraudulent",
      "raison": "Some locations have higher rates of fraud",
      "score_brut": 1.5,
      "confiance": 0.3462380168357166,
      "confiance_pct": "34.6%"
    },
    {
      "de": "Transaction_Type",
      "vers": "Fraudulent",
      "raison": "Online purchases, POS payments, bill payments, bank transfers, and ATM withdrawals can potentially involve fraud",
      "score_brut": 1.0,
      "confiance": 0.12737384814583186,
      "confiance_pct": "12.7%"
    },
    {
      "de": "Device_Used",
      "vers": "Fraudulent",
      "raison": "Certain devices might be more susceptible to fraud",
      "score_brut": 1.0,
      "confiance": 0.12737384814583186,
      "confiance_pct": "12.7%"
    },
    {
      "de": "Payment_Method",
      "vers": "Fraudulent",
      "raison": "Different payment methods may involve varying levels of risk for fraud",
      "score_brut": 1.0,
      "confiance": 0.12737384814583186,
      "confiance_pct": "12.7%"
    },
    {
      "de": "Location",
      "vers": "Transaction_Type",
      "raison": "Different locations might prefer different transaction types",
      "score_brut": 0.75,
      "confiance": 0.07725614414602817,
      "confiance_pct": "7.7%"
    },
    {
      "de": "Location",
      "vers": "Device_Used",
      "raison": "Different devices might be preferred in certain locations",
      "score_brut": 0.75,
      "confiance": 0.07725614414602817,
      "confiance_pct": "7.7%"
    },
    {
      "de": "Device_Used",
      "vers": "Payment_Method",
      "raison": "Different devices may prefer certain payment methods",
      "score_brut": 0.5,
      "confiance": 0.04685822007574478,
      "confiance_pct": "4.7%"
    },
    {
      "de": "Time_of_Transaction",
      "vers": "Fraudulent",
      "raison": "Peak hours or specific times might have higher chances of fraud",
      "score_brut": 0.02110519504151239,
      "confiance": 0.017981381088308376,
      "confiance_pct": "1.8%"
    },
    {
      "de": "Number_of_Transactions_Last_24H",
      "vers": "Fraudulent",
      "raison": "Accounts with a higher number of transactions in the last 24 hours might be riskier",
      "score_brut": 0.007753170989822537,
      "confiance": 0.01750756004939359,
      "confiance_pct": "1.8%"
    },
    {
      "de": "Time_of_Transaction",
      "vers": "Account_Age",
      "raison": "Older accounts may have more transactions during certain times",
      "score_brut": 0.004582714537434911,
      "confiance": 0.017396897357975248,
      "confiance_pct": "1.7%"
    },
    {
      "de": "Account_Age",
      "vers": "Number_of_Transactions_Last_24H",
      "raison": "Older accounts may have more transactions",
      "score_brut": 0.004214539406961192,
      "confiance": 0.01738409186330931,
      "confiance_pct": "1.7%"
    }
  ],
  "commentaire_expert": "This DAG represents a causal relationship between the given variables. Older accounts tend to have more transactions, which can potentially involve fraud. The location, transaction type, device used, payment method, time of transaction, and transaction amount all play a role in determining whether a transaction is fraudulent or not."
}

In [3]:
# Mise en forme du DAG à l'aide du package networkx

liens=dag_final["liens"]
G=nx.DiGraph()
for x in liens:
    G.add_edge(x["de"],x["vers"])


In [4]:
# Importation du dataset 

df = pd.read_csv('Fraud Detection Dataset.csv')  # changer le nom du fichier si nécessaire
print('Forme du dataset chargé :', df.shape)

Forme du dataset chargé : (51000, 12)


In [5]:

def missing_mean(df):
    """
    entrée : dataset df avec valeurs manquantes Na 
    sortie :df_mean dataset traité 
    """

    # Séparation des colonnes numériques et catégorielles
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('Fraudulent')
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns

    # Création d'une copie de sécurité
    df_c = df.copy()
    
    # Remplacement des valeurs manquantes pour les variables numériques
    for col in numeric_cols:
        for fraud_value in [0, 1]:
            mean_value = df_c.loc[df_c['Fraudulent'] == fraud_value, col].mean()
            df_c.loc[(df_c['Fraudulent'] == fraud_value) & (df_c[col].isna()),col] = mean_value
    
    # Remplacement des valeurs manquantes pour les variables catégorielles
    for col in categorical_cols:
        for fraud_value in [0, 1]:
            mode_series = df_c.loc[df_c['Fraudulent'] == fraud_value, col].mode()
            if not mode_series.empty:
                mode_value = mode_series.iloc[0]
                df_c.loc[(df_c['Fraudulent'] == fraud_value) & (df_c[col].isna()),col] = mode_value
    
    # Vérification du remplissage
    print("Pourcentage de valeurs manquantes après traitement :")
    print((df_c.isnull().mean() * 100).round(3))

    return df_c


In [6]:
df=missing_mean(df)

Pourcentage de valeurs manquantes après traitement :
Transaction_ID                      0.0
User_ID                             0.0
Transaction_Amount                  0.0
Transaction_Type                    0.0
Time_of_Transaction                 0.0
Device_Used                         0.0
Location                            0.0
Previous_Fraudulent_Transactions    0.0
Account_Age                         0.0
Number_of_Transactions_Last_24H     0.0
Payment_Method                      0.0
Fraudulent                          0.0
dtype: float64


/tmp/ipykernel_51725/984645087.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3002.9993194473086' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_c.loc[(df_c['Fraudulent'] == fraud_value) & (df_c[col].isna()),col] = mean_value
/tmp/ipykernel_51725/984645087.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.9953598680140234' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_c.loc[(df_c['Fraudulent'] == fraud_value) & (df_c[col].isna()),col] = mean_value
/tmp/ipykernel_51725/984645087.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '59.98537842854197' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_c.

In [28]:
# ── SCR : Régression Logistique / Linéaire par nœud enfant ───────────────────
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

def run_scr(dag, df, colonnes_exclues=None):
    """
    Structural Causal Regression (SCR) :
    Pour chaque nœud enfant du DAG, entraîne une régression
    sur ses nœuds parents comme features.

    - Nœud binaire (0/1)   → Régression Logistique  → métrique : AUC
    - Nœud continu/multi   → Régression Linéaire    → métrique : R²

    Retourne un DataFrame récapitulatif des résultats par nœud.
    """
    if colonnes_exclues is None:
        colonnes_exclues = ["Transaction_ID", "User_ID"]

    # Préparer le dataset : encodage des catégorielles
    df_enc = df.drop(columns=[c for c in colonnes_exclues if c in df.columns]).copy()
    for col in df_enc.select_dtypes(include=["object", "category"]).columns:
        df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

    # Nœuds enfants = tout nœud ayant au moins un parent
    noeuds_enfants = [n for n in G.nodes() if G.in_degree(n) > 0 and n in df_enc.columns]

    resultats = []
    print("=" * 70)
    print("  RÉGRESSIONS CAUSALES STRUCTURELLES (SCR)")
    print("  Nœud enfant  ←  Parents  |  Modèle  |  Métrique")
    print("=" * 70)

    for enfant in noeuds_enfants:
        parents = [p for p in G.predecessors(enfant) if p in df_enc.columns]
        if not parents:
            continue

        # Sous-dataset sans NA sur les colonnes concernées
        cols_used = parents + [enfant]
        df_sub = df_enc[cols_used].dropna()
        if len(df_sub) < 50:
            print(f"  ⚠️  {enfant} : trop peu d'observations ({len(df_sub)}), ignoré")
            continue

        X = df_sub[parents].values
        y = df_sub[enfant].values

        # Normalisation
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Choix du modèle selon le type du nœud enfant
        n_uniq = len(np.unique(y))
        is_binary = (n_uniq == 2)
        is_multiclass = (n_uniq <= 10 and n_uniq > 2)
        is_continuous = (n_uniq > 10)

        if is_binary:
            model = LogisticRegression(max_iter=500, random_state=42)
            model.fit(X_scaled, y)
            y_proba = model.predict_proba(X_scaled)[:, 1]
            metric_val = roc_auc_score(y, y_proba)
            metric_name = "AUC"
            model_type = "LogReg (binaire)"
            coefs = dict(zip(parents, model.coef_[0].round(4)))
            family = sm.families.Binomial()

        elif is_multiclass:
            model = LogisticRegression(max_iter=500, random_state=42)
            model.fit(X_scaled, y)
            y_pred = model.predict(X_scaled)
            metric_val = np.mean(y_pred == y)  # accuracy
            metric_name = "Accuracy"
            model_type = "LogReg (multi)"
            coefs = dict(zip(parents, model.coef_[0].round(4)))
            family = sm.families.Gaussian()

        else:
            model = LinearRegression()
            model.fit(X_scaled, y)
            y_pred = model.predict(X_scaled)
            metric_val = r2_score(y, y_pred)
            metric_name = "R²"
            model_type = "LinReg (continu)"
            coefs = dict(zip(parents, model.coef_.round(4)))
            family = sm.families.Gaussian()

        print(f"\n  [{enfant}]  ←  {parents}")
        print(f"    Modèle    : {model_type}")
        print(f"    {metric_name:10}: {metric_val:.4f}")
        print(f"    Coefs     : {coefs}")

        resultats.append({
            "Nœud enfant":  enfant,
            "Parents":      ", ".join(parents),
            "Modèle":       model_type,
            "Métrique":     metric_name,
            "Score":        round(metric_val, 4),
            "Coefficients": coefs,
            "famille": family,
            "formule" : "+".join(parents)
        })

    print("\n" + "=" * 70)
    df_scr = pd.DataFrame(resultats)
    return df_scr,resultats


# ── Exécution sur le DAG final validé ────────────────────────────────────────
df_scr,result = run_scr(dag_final, df, colonnes_exclues=None)
print("\nRécapitulatif SCR :")
print(df_scr[["Nœud enfant", "Parents", "Modèle", "Métrique", "Score"]].to_string(index=False))

  RÉGRESSIONS CAUSALES STRUCTURELLES (SCR)
  Nœud enfant  ←  Parents  |  Modèle  |  Métrique

  [Fraudulent]  ←  ['Location', 'Transaction_Type', 'Device_Used', 'Payment_Method', 'Time_of_Transaction', 'Number_of_Transactions_Last_24H']
    Modèle    : LogReg (binaire)
    AUC       : 0.5197
    Coefs     : {'Location': np.float64(-0.0183), 'Transaction_Type': np.float64(0.0326), 'Device_Used': np.float64(0.0496), 'Payment_Method': np.float64(0.0068), 'Time_of_Transaction': np.float64(0.0333), 'Number_of_Transactions_Last_24H': np.float64(-0.0176)}

  [Transaction_Type]  ←  ['Location']
    Modèle    : LogReg (multi)
    Accuracy  : 0.2042
    Coefs     : {'Location': np.float64(-0.0045)}

  [Device_Used]  ←  ['Location']
    Modèle    : LogReg (multi)
    Accuracy  : 0.3552
    Coefs     : {'Location': np.float64(-0.0067)}

  [Payment_Method]  ←  ['Device_Used']
    Modèle    : LogReg (multi)
    Accuracy  : 0.2815
    Coefs     : {'Device_Used': np.float64(-0.0156)}

  [Number_of_Tra

In [29]:
# Préparer le dataset : encodage des catégorielles
colonnes_exclues= ["Transaction_ID", "User_ID"]
df_enc = df.drop(columns=[c for c in colonnes_exclues if c in df.columns]).copy()
for col in df_enc.select_dtypes(include=["object", "category"]).columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

In [30]:
# Nœuds enfants = tout nœud ayant au moins un parent
noeuds_enfants = [n for n in G.nodes() if G.in_degree(n) > 0 and n in df_enc.columns]
noeuds_enfants

['Fraudulent',
 'Transaction_Type',
 'Device_Used',
 'Payment_Method',
 'Number_of_Transactions_Last_24H',
 'Account_Age']

In [31]:
# Ajustement DAG ordre topologique

carac = {cle : 0 for cle in noeuds_enfants}
for res in result:
    link=0
    if res["Modèle"]=="LogReg (multi)":
        link=sm.families.links.Log()
    else:
        link=sm.families.links.Logit()
    carac[res["Nœud enfant"]]=[res["famille"],link,res["formule"]]

carac
    

{'Fraudulent': [<statsmodels.genmod.families.family.Binomial at 0x7f7531416850>,
  'Location+Transaction_Type+Device_Used+Payment_Method+Time_of_Transaction+Number_of_Transactions_Last_24H'],
 'Transaction_Type': [<statsmodels.genmod.families.family.Gaussian at 0x7f7531e257f0>,
  'Location'],
 'Device_Used': [<statsmodels.genmod.families.family.Gaussian at 0x7f7531b908d0>,
  'Location'],
 'Payment_Method': [<statsmodels.genmod.families.family.Gaussian at 0x7f7531d87ac0>,
  'Device_Used'],
 'Number_of_Transactions_Last_24H': [<statsmodels.genmod.families.family.Gaussian at 0x7f75273eb150>,
  'Account_Age'],
 'Account_Age': [<statsmodels.genmod.families.family.Gaussian at 0x7f752722ed50>,
  'Time_of_Transaction']}

In [32]:
order=list(nx.topological_sort(G))
order

['Location',
 'Time_of_Transaction',
 'Transaction_Type',
 'Device_Used',
 'Account_Age',
 'Payment_Method',
 'Number_of_Transactions_Last_24H',
 'Fraudulent']

In [23]:
import statsmodels.api as sm
import pandas as pd


y = df['Payment_Method']
explic = ['Device_Used']

# Encodage des variables catégorielles
X = pd.get_dummies(df[explic], drop_first=True, dtype=float)

# On ajoute l'intercept
X = sm.add_constant(X)

# 3. On crée et ajuste le modèle
model = sm.MNLogit(y, X)
result = model.fit(maxiter=300, method='bfgs')

print(result.summary())


Optimization terminated successfully.
         Current function value: 1.475242
         Iterations: 88
         Function evaluations: 89
         Gradient evaluations: 89
                          MNLogit Regression Results                          
Dep. Variable:         Payment_Method   No. Observations:                51000
Model:                        MNLogit   Df Residuals:                    50984
Method:                           MLE   Df Model:                           12
Date:                Wed, 22 Apr 2026   Pseudo R-squ.:               0.0001025
Time:                        09:25:58   Log-Likelihood:                -75237.
converged:                       True   LL-Null:                       -75245.
Covariance Type:            nonrobust   LLR p-value:                    0.2191
    Payment_Method=Debit Card       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------


In [33]:
# Création des GLM pour chaque noeud enfant

glms={}
for noeud in noeuds_enfants:
    par=carac[noeud][2]
    formule=f"{noeud} ~ {par}"
    if isinstance(carac[noeud][0],sm.families.Gaussian):
        y=df[noeud]
        explic=[x.strip() for x in carac['Fraudulent'][2].split('+')]
        X = pd.get_dummies(df[explic], drop_first=True, dtype=float)
        X = sm.add_constant(X)
        mod=sm.MNLogit(y, X)
        modele = mod.fit(maxiter=300, method='bfgs')
    else:
        mod=smf.glm(formula=formule, data=df, family=sm.families.Binomial(sm.families.links.Logit()))
        modele=mod.fit()
    glms[noeud]= modele

Optimization terminated successfully.
         Current function value: 0.000001
         Iterations: 31
         Function evaluations: 36
         Gradient evaluations: 36
Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 32
         Function evaluations: 36
         Gradient evaluations: 36
Optimization terminated successfully.
         Current function value: 0.000005
         Iterations: 30
         Function evaluations: 34
         Gradient evaluations: 34
         Current function value: nan
         Iterations: 150
         Function evaluations: 266
         Gradient evaluations: 266
         Current function value: 4.754031
         Iterations: 300
         Function evaluations: 301
         Gradient evaluations: 301


In [34]:
glms

{'Fraudulent': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x7f7531df3380>,
 'Transaction_Type': <statsmodels.discrete.discrete_model.MultinomialResultsWrapper at 0x7f7531b91040>,
 'Device_Used': <statsmodels.discrete.discrete_model.MultinomialResultsWrapper at 0x7f7531b916a0>,
 'Payment_Method': <statsmodels.discrete.discrete_model.MultinomialResultsWrapper at 0x7f7531423550>,
 'Number_of_Transactions_Last_24H': <statsmodels.discrete.discrete_model.MultinomialResultsWrapper at 0x7f7531423650>,
 'Account_Age': <statsmodels.discrete.discrete_model.MultinomialResultsWrapper at 0x7f7531a24140>}